In [4]:
import cv2

cap = cv2.VideoCapture("C:/Users/gmission/Desktop/video-test/GX012760.MP4")
ret, frame = cap.read()
cap.release()

points = []

def click(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        points.append([x, y])
        cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)   # Red dot
        if len(points) > 1:
            cv2.line(frame, tuple(points[-2]), tuple(points[-1]), (0, 255, 0), 2)   # Green line
        cv2.imshow("Draw Zone - Press Q when done", frame)

cv2.imshow("Draw Zone - Press Q when done", frame)
cv2.setMouseCallback("Draw Zone - Press Q when done", click)
cv2.waitKey(0)
cv2.destroyAllWindows()

print(f"Zone polygon: {points}")

Zone polygon: [[926, 419], [1337, 522], [918, 786], [692, 641]]


In [5]:
import cv2
import sys
sys.path.append("../ai/src")

from ultralytics import YOLO
from events.intrusion import Zone, IntrusionDetector

model = YOLO("yolo11n.pt")

zone = Zone("test_area", [[926, 419], [1337, 522], [918, 786], [692, 641]])
detector = IntrusionDetector([zone])

cap = cv2.VideoCapture("C:/Users/gmission/Desktop/video-test/GX012760.MP4")

for i in range(500):    # Skip first 500 frames
    cap.read()

ret, frame = cap.read()
if ret:
    results = model.track(frame, persist=True, classes=[0])
    events = detector.check(results)
    print(f"Events: {events}")

    for box in results[0].boxes:
        name = results[0].names[int(box.cls[0])]
        print(f" {name}, id: {box.id}")

cap.release()


0: 384x640 1 person, 5.1ms
Speed: 1.1ms preprocess, 5.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)
Events: []
 person, id: tensor([1.])


In [ ]:
import cv2
from ultralytics import YOLO
from events.intrusion import Zone, IntrusionDetector
import numpy as np

model = YOLO("yolo11n.pt")
zone = Zone("test_area", [[926, 419], [1337, 522], [918, 786], [692, 641]])
detector = IntrusionDetector([zone])

cap = cv2.VideoCapture("C:/Users/gmission/Desktop/video-test/GX012760.MP4")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, conf=0.5, classes=[0])

    # Draw tracking results
    annotated = results[0].plot(line_width=1, font_size=0.5)

    # Draw zone on annotated frame
    pts = zone.polygon.reshape((-1, 1, 2))
    cv2.polylines(annotated, [pts], True, (0, 255, 0), 2)

    # Check intrusion
    events = detector.check(results)
    for event in events:
        cv2.putText(annotated, f"INTRUSION: ID {event['track_id']}",
                    (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 2)
        
    cv2.imshow("Intrusion Detection", annotated)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 (no detections), 9.4ms
Speed: 1.1ms preprocess, 9.4ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 5.7ms
Speed: 1.7ms preprocess, 5.7ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 5.7ms
Speed: 1.1ms preprocess, 5.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 13.3ms
Speed: 1.2ms preprocess, 13.3ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.8ms
Speed: 0.9ms preprocess, 4.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 5.0ms
Speed: 1.0ms preprocess, 5.0ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 5.6ms
Speed: 1.0ms preprocess, 5.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 10.4ms
Speed: 1.1ms preprocess, 10.4ms inference, 1.